# PPO VM Allocation Experiments

This notebook trains and evaluates PPO agents for VM allocation using the existing DRL pipeline:

- `rl/environment.py`: `VMAllocationEnv` (Gymnasium environment)
- `rl/config.py`: PPO and reward configuration
- `train_ppo.py`: training script
- `eval_ppo.py`: evaluation and comparison with LP baseline

You can use this notebook to:
- Run quick training experiments (e.g., smaller `total_timesteps`)
- Evaluate PPO against the LP baseline
- Inspect the generated PPO schedules and metrics.


In [ ]:
# Imports and configuration

from pathlib import Path

from rl.config import PPOConfig, SCENARIO_OVERLOAD, SCENARIO_COST
from train_ppo import train_ppo
from eval_ppo import evaluate_scenario, print_comparison

# Ensure latest code is loaded if the notebook stays open while editing
import importlib
importlib.reload(train_ppo)
importlib.reload(eval_ppo)
importlib.reload(__import__('rl.config').config)

# Refresh config after reload
from rl.config import PPOConfig, SCENARIO_OVERLOAD, SCENARIO_COST

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

# Show current default PPO configuration
config = PPOConfig()
config


Project root: e:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure


PPOConfig(learning_rate=0.0003, n_steps=2048, batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2, clip_range_vf=None, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5, episode_length=480, horizon=18, total_timesteps=500000, tensorboard_log='./tensorboard_logs/', log_interval=10, save_freq=10000)

In [ ]:
# Quick training config (adjust as needed)
from copy import deepcopy

# Number of parallel envs (increase if you have CPU/GPU resources)
N_ENVS = 8

exp_config = deepcopy(config)

# Default: 1,000,000 timesteps (recommended after reward tweaks)
# For a quick debug run, set to 50_000
exp_config.total_timesteps = 1_000_000

print("Training configuration override:")
print("  total_timesteps =", exp_config.total_timesteps)
print("  episode_length  =", exp_config.episode_length)
print("  horizon         =", exp_config.horizon)
print("  n_envs          =", N_ENVS)

In [ ]:

print("Training configuration:")
print("  total_timesteps =", exp_config.total_timesteps)
print("  episode_length  =", exp_config.episode_length)
print("  horizon         =", exp_config.horizon)
print("  n_envs          =", N_ENVS)

# Train overload-first scenario
model_overload = train_ppo(
    scenario=SCENARIO_OVERLOAD,
    config=exp_config,
    total_timesteps=exp_config.total_timesteps,
    n_envs=N_ENVS,
    continue_training=False,
)

# Train cost-first scenario
model_cost = train_ppo(
    scenario=SCENARIO_COST,
    config=exp_config,
    total_timesteps=exp_config.total_timesteps,
    n_envs=N_ENVS,
    continue_training=False,
)


Training configuration:
  total_timesteps = 50000
  episode_length  = 480
  horizon         = 18

Training PPO for scenario: OVERLOAD
Creating new PPO model
Using cpu device

Starting training for 50,000 timesteps...
Logging to ./tensorboard_logs/PPO_7


Output()

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 480      |
|    ep_rew_mean     | -579     |
| time/              |          |
|    fps             | 467      |
|    iterations      | 1        |
|    time_elapsed    | 17       |
|    total_timesteps | 8192     |
---------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 480        |
|    ep_rew_mean          | -575       |
| time/                   |            |
|    fps                  | 426        |
|    iterations           | 2          |
|    time_elapsed         | 38         |
|    total_timesteps      | 16384      |
| train/                  |            |
|    approx_kl            | 0.01831323 |
|    clip_fraction        | 0.196      |
|    clip_range           | 0.2        |
|    entropy_loss         | -9.58      |
|    explained_variance   | -0.328     |
|    learning_rate        | 0.0003     |
|    loss                 | -0.0931    |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.0385    |
|    value_loss           | 0.214      |
----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 480         |
|    ep_rew_mean          | -570        |
| time/                   |             |
|    fps                  | 417         |
|    iterations           | 3           |
|    time_elapsed         | 58          |
|    total_timesteps      | 24576       |
| train/                  |             |
|    approx_kl            | 0.019599698 |
|    clip_fraction        | 0.218       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.56       |
|    explained_variance   | 0.44        |
|    learning_rate        | 0.0003      |
|    loss                 | -0.153      |
|    n_updates            | 20          |
|    policy_gradient_loss | -0.0441     |
|    value_loss           | 0.0269      |
-----------------------------------------


...

New best mean reward!

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 480       |
|    ep_rew_mean     | -4.84e+03 |
| time/              |           |
|    fps             | 425       |
|    iterations      | 5         |
|    time_elapsed    | 96        |
|    total_timesteps | 40960     |
----------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 480         |
|    ep_rew_mean          | -4.75e+03   |
| time/                   |             |
|    fps                  | 435         |
|    iterations           | 6           |
|    time_elapsed         | 112         |
|    total_timesteps      | 49152       |
| train/                  |             |
|    approx_kl            | 0.021597862 |
|    clip_fraction        | 0.27        |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.44       |
|    explained_variance   | 0.36        |
|    learning_rate        | 0.0003      |
|    loss                 | -0.13       |
|    n_updates            | 50          |
|    policy_gradient_loss | -0.0508     |
|    value_loss           | 0.0084      |
-----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 480        |
|    ep_rew_mean          | -4.57e+03  |
| time/                   |            |
|    fps                  | 434        |
|    iterations           | 7          |
|    time_elapsed         | 131        |
|    total_timesteps      | 57344      |
| train/                  |            |
|    approx_kl            | 0.02336258 |
|    clip_fraction        | 0.301      |
|    clip_range           | 0.2        |
|    entropy_loss         | -9.38      |
|    explained_variance   | -0.0734    |
|    learning_rate        | 0.0003     |
|    loss                 | -0.166     |
|    n_updates            | 60         |
|    policy_gradient_loss | -0.0549    |
|    value_loss           | 0.00719    |
----------------------------------------



Training completed in 0:02:16.883749
Model saved to E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_cost.zip
VecNormalize saved to E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_cost_vecnormalize.pkl


In [4]:
# Evaluation: run PPO on test set and compare with LP baseline

results = {}

for scenario in [SCENARIO_OVERLOAD, SCENARIO_COST]:
    comp = evaluate_scenario(scenario)
    results[scenario] = comp
    print_comparison(comp)

results



Evaluating scenario: OVERLOAD
Loaded model from E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_overload.zip

Running PPO rollout for scenario: overload
PPO schedule saved to E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\ppo_schedule_test_overload.csv
Run 'python vm_resource_planner.py' first to generate baseline.

RESULTS: OVERLOAD

PPO Performance:
  Total VM Cost: $1739.4976
  Total Switching Cost: $131.5200
  Total Cost: $1871.0176
  SLA Violations: 0 (0.0%)
  Mean CPU Utilization: 0.0%
  Mean Memory Utilization: 0.0%

Evaluating scenario: COST
Loaded model from E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_cost.zip

Running PPO rollout for scenario: cost
PPO schedule saved to E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\ppo_schedule_test_cost.csv
Run 'python vm_resource_planner.py' fi

{'overload': {'scenario': 'overload',
  'timestamp': '2025-12-09 23:06:58',
  'ppo': {'total_vm_cost': 1739.4976,
   'total_switching_cost': 131.52,
   'total_cost': 1871.0176,
   'sla_violations': 0,
   'sla_violation_rate': 0.0,
   'mean_cpu_utilization': 0.012234401981969345,
   'mean_mem_utilization': 0.0,
   'n_steps': 480}},
 'cost': {'scenario': 'cost',
  'timestamp': '2025-12-09 23:06:59',
  'ppo': {'total_vm_cost': 1326.1568,
   'total_switching_cost': 122.29,
   'total_cost': 1448.4468000000002,
   'sla_violations': 0,
   'sla_violation_rate': 0.0,
   'mean_cpu_utilization': 0.03188786001823282,
   'mean_mem_utilization': 0.0,
   'n_steps': 480}}}

In [ ]:
# Inspect generated PPO schedules and comparison JSON

import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("forecast_result")

ppo_overload_path = RESULTS_DIR / "ppo_schedule_test_overload.csv"
ppo_cost_path = RESULTS_DIR / "ppo_schedule_test_cost.csv"
comparison_path = RESULTS_DIR / "ppo_vs_lp_comparison.json"

print("PPO overload schedule exists:", ppo_overload_path.exists())
print("PPO cost schedule exists:", ppo_cost_path.exists())
print("Comparison JSON exists:", comparison_path.exists())

if ppo_overload_path.exists():
    df_over = pd.read_csv(ppo_overload_path)
    display(df_over.head())

if ppo_cost_path.exists():
    df_cost = pd.read_csv(ppo_cost_path)
    display(df_cost.head())

# Show 30-minute bucketed outputs (align with LP)
ppo_overload_bucket = RESULTS_DIR / "ppo_schedule_30min_overload.csv"
ppo_cost_bucket = RESULTS_DIR / "ppo_schedule_30min_cost.csv"
print("PPO overload 30min bucket exists:", ppo_overload_bucket.exists())
print("PPO cost 30min bucket exists:", ppo_cost_bucket.exists())

if ppo_overload_bucket.exists():
    df_over_b = pd.read_csv(ppo_overload_bucket)
    display(df_over_b.head())

if ppo_cost_bucket.exists():
    df_cost_b = pd.read_csv(ppo_cost_bucket)
    display(df_cost_b.head())

if comparison_path.exists():
    import json
    with open(comparison_path, "r") as f:
        comp_data = json.load(f)
    comp_data


PPO overload schedule exists: True
PPO cost schedule exists: True
Comparison JSON exists: False


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,"D2s_v3×8, D8s_v3×1",1.152,0.21,1.362,24.7,131.0,24.0,128.0,0.24,0.258175,0.0,0,0.0,0.0,0
1,1970-01-25 01:06:00,"D2s_v3×8, D8s_v3×1",1.152,0.00,1.152,24.7,131.0,24.0,128.0,0.22,0.258186,0.0,0,0.0,0.0,0
2,1970-01-25 01:06:30,"D2s_v3×8, D8s_v3×1",1.152,0.00,1.152,24.7,131.0,24.0,128.0,0.13,0.258044,0.0,0,0.0,0.0,0
3,1970-01-25 01:07:00,"D2s_v3×8, D8s_v3×1",1.152,0.00,1.152,24.7,131.0,24.0,128.0,0.08,0.257840,0.0,0,0.0,0.0,0
4,1970-01-25 01:07:30,"D2s_v3×8, D8s_v3×1",1.152,0.00,1.152,24.7,131.0,24.0,128.0,0.10,0.258327,0.0,0,0.0,0.0,0


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,"B2s×3, D2s_v3×8, D8s_v3×2",1.6608,0.29,1.9508,38.7,207.0,38.0,204.0,0.24,0.258175,0.0,0,0.0,0.0,0
1,1970-01-25 01:06:00,"B2s×1, D2s_v3×8, D8s_v3×2, D32s_v3×1",3.1136,0.12,3.2336,66.7,327.0,66.0,324.0,0.22,0.258186,0.0,0,0.0,0.0,0
2,1970-01-25 01:06:30,"B2s×1, D2s_v3×8, D32s_v3×1",2.3456,0.10,2.4456,50.7,199.0,50.0,196.0,0.13,0.258044,0.0,0,0.0,0.0,0
3,1970-01-25 01:07:00,"B2s×3, D2s_v3×8, D8s_v3×2, D32s_v3×1",3.1968,0.12,3.3168,70.7,335.0,70.0,332.0,0.08,0.257840,0.0,0,0.0,0.0,0
4,1970-01-25 01:07:30,"D2s_v3×8, D8s_v3×2, D32s_v3×1",3.0720,0.03,3.1020,64.7,323.0,64.0,320.0,0.10,0.258327,0.0,0,0.0,0.0,0
